# Paper Number Verifier — *Terms of Engagement in Constitutions*

Reproducibility audit for the paper (Powell & Bambrick). Recomputes **every reported
number in the paper** from the committed source data and asserts each matches.

**No setup, no GPU, ~30 seconds.**

### How to run

1. **Cell 1** — get the bundle into Colab. Two options:
   - **Option A (Google Drive):** mount Drive and point to `constitutions-reproducibility.tar.gz` on your Drive.
   - **Option B (Upload):** upload the tarball directly from your computer.
2. **Cell 2** — unpack the bundle and install dependencies.
3. **Cell 3** — run all 62 checks. Expected output: `62/62 checks passed`.

Everything needed (the coded provision data, similarity matrices, network files,
and all QAP/MRQAP/ERGM result files) is inside the tarball. You do not upload any
individual data files.

In [ ]:
#@title Cell 1 — Get the reproducibility bundle into Colab
#@markdown Choose how to provide `constitutions-reproducibility.tar.gz`, then run this cell.
import os, shutil

SOURCE = "Upload from computer"  #@param ["Upload from computer", "Google Drive"]
#@markdown If using Google Drive, set the path to the tarball on your Drive:
DRIVE_PATH = "/content/drive/MyDrive/constitutions-reproducibility.tar.gz"  #@param {type:"string"}

TARBALL = "/content/constitutions-reproducibility.tar.gz"

if SOURCE == "Google Drive":
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists(DRIVE_PATH):
        raise FileNotFoundError(
            f'Tarball not found at {DRIVE_PATH}. '
            'Check the path or use the Upload option.')
    shutil.copy(DRIVE_PATH, TARBALL)
    print(f'Copied from Drive: {DRIVE_PATH}')
else:
    from google.colab import files
    print('Upload constitutions-reproducibility.tar.gz:')
    up = files.upload()
    name = list(up.keys())[0]
    if name != os.path.basename(TARBALL):
        shutil.move(name, TARBALL)
    print(f'Uploaded: {name}')

assert os.path.exists(TARBALL), 'Tarball not present.'
print(f'\nBundle ready: {TARBALL} ({os.path.getsize(TARBALL)} bytes)')

In [ ]:
#@title Cell 2 — Unpack the bundle and install dependencies
import tarfile, subprocess, sys, os

BUNDLE_DIR = "/content/verifier"
os.makedirs(BUNDLE_DIR, exist_ok=True)

with tarfile.open("/content/constitutions-reproducibility.tar.gz") as tf:
    tf.extractall(BUNDLE_DIR)

print('Unpacked contents:')
for root, dirs, fnames in os.walk(BUNDLE_DIR):
    for fn in sorted(fnames):
        rel = os.path.relpath(os.path.join(root, fn), BUNDLE_DIR)
        print(f'  {rel}')

print('\nInstalling dependencies...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                os.path.join(BUNDLE_DIR, 'requirements.txt')], check=True)
print('Dependencies installed. Proceed to Cell 3.')

In [ ]:
#@title Cell 3 — Run all 62 checks
import sys, importlib
sys.path.insert(0, '/content/verifier')

# (re)load the verifier module from the bundle
import verifier_core
importlib.reload(verifier_core)
from verifier_core import run_all_checks

passed, failed, results = run_all_checks(verbose=True)

print('\nSummary by section:')
sections = {}
for cid, sec, status, paper, val, claim in results:
    sections.setdefault(sec, [0, 0])
    if status == 'PASS': sections[sec][0] += 1
    else: sections[sec][1] += 1
for sec in sorted(sections):
    p, f = sections[sec]
    flag = '' if f == 0 else f'  <-- {f} FAILED'
    print(f'  {sec:<20} {p} passed, {f} failed{flag}')

print()
print(f'{"SUCCESS" if failed==0 else "FAILURE"}: {passed}/{passed+failed} checks passed.')